# Full lifespan — what happens when SoH drops to 0?

Pushes the PINN's prediction horizon out to **very long cycle counts** (10 k, 50 k, 200 k, 1 M) to see whether the trajectory ever reaches **SoH = 0** — or asymptotes somewhere short of zero.

Why this might *not* reach zero: the Neural ODE in [`src/pinn/model.py`](../src/pinn/model.py) enforces `dSoH/dn = −softplus(NN(SoH, n, x_health))`. The constraint guarantees a monotone decrease, but the *magnitude* of the decrease is set by the network output. After fine-tuning on cells that only aged to SoH ≈ 0.86, the network never learned to produce large fade rates — so the trajectory tends to asymptote rather than cross zero.

Workflow:
1. Load cached REPT cycle data, robust normalisation + trim.
2. Sweep horizons 10 k → 50 k → 200 k → 1 M cycles per cell.
3. For each cell, record where (if ever) the predicted SoH crosses key thresholds 0.80, 0.60, 0.40, 0.20, 0.05, 0.0.
4. Plot a representative cell's full lifespan on linear and semi-log axes, plus the instantaneous fade rate `−dSoH/dn`.
5. Per-cell overview at the largest horizon.

Expectation up-front: the model is structurally incapable of reaching SoH = 0 because the fade rate decays as the NN's input SoH shrinks. The plots below quantify exactly where the asymptote sits.

In [ ]:
from __future__ import annotations
import os, sys, time
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if (HERE.parent / 'CLAUDE.md').exists() else HERE
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.inference.predict_rul import load_model, predict_rul_with_uncertainty

np.random.seed(456); torch.manual_seed(456)
model = load_model(ckpt_path=Path('outputs/models/pinn_finetuned.pt'))
print(f'Model: pinn_finetuned.pt, {model.n_parameters():,} params')

HORIZONS = [10_000, 50_000, 200_000, 1_000_000]
THRESHOLDS = [0.80, 0.60, 0.40, 0.20, 0.10, 0.05, 0.01, 0.0]
N_MC = 100   # fewer MC samples since we run several horizons
N_PTS = 600   # eval grid

## 1. Load cached REPT cycle data + robust normalisation

Same per-cell normalisation as notebook 06: reference = median dchg cap over cycles 5–30; drop SoH outside `[0.70, 1.10]`.

In [ ]:
df = pd.read_csv('Lab_Data_Cell/.cached_rept_cycle_for_pinn.csv', dtype={'cell':str})
df['cell'] = df['cell'].astype(str).str.zfill(4)
df = df.sort_values(['cell','batch','cycle_no']).reset_index(drop=True)
df['global_cycle'] = df.groupby('cell').cumcount() + 1

def parse_crate(s):
    if s is None or pd.isna(s): return float('nan')
    try: return float(str(s).rstrip('CcDd '))
    except ValueError: return float('nan')

def normalise_cell(g):
    ref = g[(g['global_cycle']>=5) & (g['global_cycle']<=30)]
    if len(ref) < 5: ref = g.head(20)
    g = g.assign(SoH=g['dchg_cap_ah']/float(ref['dchg_cap_ah'].median()))
    good = g[(g['SoH']>=0.70) & (g['SoH']<=1.10)].sort_values('global_cycle').reset_index(drop=True)
    good['cycle_trim'] = range(1, len(good)+1)
    return good

def cell_inputs(cid):
    g = df[df['cell']==cid]
    early = g[g['global_cycle']<=10]
    return {
        'cell': cid,
        'dcir_mOhm': float(early['ir_ohm'].median())*1000.0,
        'crate': parse_crate(g['crate'].iloc[0]),
        'dod': str(g['dod'].iloc[0]),
        'good': normalise_cell(g),
    }

info = {cid: cell_inputs(cid) for cid in sorted(df['cell'].unique())}
ready = [cid for cid, v in info.items() if len(v['good']) >= 30 and v['crate']==v['crate']]
print(f'{len(ready)} REPT cells eligible.')

## 2. Horizon sweep — what SoH does the model reach?

For each cell, run the prediction at every horizon. For each horizon, record `SoH(horizon)` (the trajectory endpoint) and the **first** cycle (if any) where the mean trajectory crosses each threshold in `THRESHOLDS`.

In [ ]:
def predict(cid, horizon):
    v = info[cid]
    x = np.array([25.0, v['crate'], v['dcir_mOhm'], 0.0, 1.0], dtype=np.float32)
    return predict_rul_with_uncertainty(
        model=model, soh_now=1.0, cycle_now=0.0, x_health=x,
        n_samples=N_MC, feature_noise_std=0.01,
        eol=0.0, max_cycles=horizon, n_points=N_PTS)

def first_crossing(traj, n_axis, level):
    below = np.where(traj < level)[0]
    return float(n_axis[below[0]]) if len(below) else None

rows = []
for cid in ready:
    for H in HORIZONS:
        out = predict(cid, H)
        n_axis = np.array(out['n_axis'])
        s_m = np.array(out['soh_trajectory_mean'])
        row = {'cell': cid, 'horizon': H,
                'SoH_end_mean': round(float(s_m[-1]), 4),
                'SoH_min_mean': round(float(s_m.min()), 4)}
        for th in THRESHOLDS:
            c = first_crossing(s_m, n_axis, th)
            row[f'cyc_to_{th:g}'] = (round(c, 0) if c is not None else 'no crossing')
        rows.append(row)
    print(f'  {cid}: '
          f'SoH @ 10k={rows[-4]["SoH_end_mean"]:.4f}, '
          f'50k={rows[-3]["SoH_end_mean"]:.4f}, '
          f'200k={rows[-2]["SoH_end_mean"]:.4f}, '
          f'1M={rows[-1]["SoH_end_mean"]:.4f}')

sweep_df = pd.DataFrame(rows)
sweep_df.to_csv('outputs/results/pinn_horizon_sweep.csv', index=False)
print(f'\nWrote outputs/results/pinn_horizon_sweep.csv ({len(sweep_df)} rows)')

# Summary at the largest horizon
tail = sweep_df[sweep_df['horizon']==HORIZONS[-1]]
print(f'\n=== At horizon = {HORIZONS[-1]:,} cycles ===')
print(f'  min predicted SoH across all cells: {tail["SoH_end_mean"].min():.4f}')
print(f'  max predicted SoH across all cells: {tail["SoH_end_mean"].max():.4f}')
for th in THRESHOLDS:
    col = f'cyc_to_{th:g}'
    n_cross = (tail[col] != 'no crossing').sum()
    print(f'  cells crossing SoH={th}: {n_cross} / {len(tail)}')

## 3. Representative cell — full-lifespan trajectory + rate

Pick one well-behaved cell (typically the lowest-MAE 0_80 / 0.25 C cell). Plot at horizon 1 M cycles:
- top-left: SoH vs cycle (linear)
- top-right: SoH vs cycle (semi-log on cycle axis)
- bottom: fade rate `−dSoH/dn` vs cycle (also semi-log)

The rate plot shows directly why the model can't reach SoH = 0: the fade rate decays toward 0 as SoH approaches its asymptote.

In [ ]:
REP_CELL = '0004'   # 0_80, 0.25C, low DCIR — was best in the early-life MAE table
if REP_CELL not in ready:
    REP_CELL = ready[0]
v = info[REP_CELL]
print(f'Representative cell: {REP_CELL}  (DCIR {v["dcir_mOhm"]:.2f} mΩ, {v["crate"]:.2f}C, DoD {v["dod"]})')

out = predict(REP_CELL, HORIZONS[-1])
n_axis = np.array(out['n_axis'])
s_m  = np.array(out['soh_trajectory_mean'])
s_p5 = np.array(out['soh_trajectory_p5'])
s_p95 = np.array(out['soh_trajectory_p95'])

# Approx fade rate from the discrete trajectory: -d(SoH)/d(n)
rate = -np.gradient(s_m, n_axis)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
(ax_lin, ax_log), (ax_rate, ax_close) = axes

for ax, xlim in [(ax_lin, (0, HORIZONS[-1]))]:
    ax.fill_between(n_axis, s_p5, s_p95, color='C3', alpha=0.18, label='90 % CI')
    ax.plot(n_axis, s_m, '-', color='C3', lw=1.6, label='PINN mean')
    ax.plot(v['good']['cycle_trim'], v['good']['SoH'], 'ko', ms=3, alpha=0.6, label=f'measured ({len(v["good"])} pts)')
    for th in [0.80, 0.60, 0.40, 0.20, 0.0]:
        ax.axhline(th, ls='--', color='gray', alpha=0.4, lw=0.8)
        ax.text(xlim[1]*0.98, th+0.01, f'SoH {th}', fontsize=7, color='gray', ha='right')
    ax.set(xlabel='Cycle', ylabel='SoH', xlim=xlim, ylim=(-0.05, 1.05),
            title=f'Linear horizon — cell {REP_CELL} (1 M cyc)')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8, loc='upper right')

ax_log.fill_between(n_axis, s_p5, s_p95, color='C3', alpha=0.18, label='90 % CI')
ax_log.plot(n_axis, s_m, '-', color='C3', lw=1.6, label='PINN mean')
ax_log.plot(v['good']['cycle_trim'], v['good']['SoH'], 'ko', ms=3, alpha=0.6, label=f'measured')
for th in [0.80, 0.60, 0.40, 0.20, 0.0]:
    ax_log.axhline(th, ls='--', color='gray', alpha=0.4, lw=0.8)
ax_log.set_xscale('symlog', linthresh=100)
ax_log.set(xlabel='Cycle (symlog)', ylabel='SoH', ylim=(-0.05, 1.05),
            title='Semi-log (same cell)')
ax_log.grid(True, which='both', alpha=0.3); ax_log.legend(fontsize=8, loc='upper right')

ax_rate.plot(n_axis, rate, '-', color='C2', lw=1.5)
ax_rate.set_xscale('symlog', linthresh=100); ax_rate.set_yscale('log')
ax_rate.set(xlabel='Cycle (symlog)', ylabel='Fade rate  −dSoH/dn  [1/cyc]',
             title='Instantaneous fade rate — log-log')
ax_rate.grid(True, which='both', alpha=0.3)

# Close-up of the asymptote: zoom into the tail
ax_close.fill_between(n_axis, s_p5, s_p95, color='C3', alpha=0.18)
ax_close.plot(n_axis, s_m, '-', color='C3', lw=1.6, label='PINN mean')
ymin = float(min(s_p5.min(), s_m.min()) - 0.002)
ymax = float(max(s_p95.max(), s_m.max()) + 0.002)
ax_close.set(xlabel='Cycle', ylabel='SoH', xlim=(HORIZONS[-1]*0.001, HORIZONS[-1]),
              ylim=(max(0, ymin), min(1.0, ymax) if ymax > 0.001 else 0.05),
              title='Tail close-up — asymptotic SoH')
ax_close.grid(True, alpha=0.3)
ax_close.text(0.02, 0.95, f'end SoH = {s_m[-1]:.4f}\nmin SoH over horizon = {s_m.min():.4f}',
               transform=ax_close.transAxes, va='top', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.suptitle(f'PINN full-lifespan analysis — REPT cell {REP_CELL} (1 M cycles)', fontsize=13)
fig.tight_layout()
out_path = Path('outputs/results/pinn_full_lifespan_representative.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'wrote → {out_path}')

## 4. All cells at 1 M cycles

One overview figure with every cell's mean trajectory on a symlog cycle axis. Lets you see at a glance whether *any* cell's trajectory ever bends down toward zero.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.0))
cmap = plt.cm.viridis(np.linspace(0, 1, len(ready)))
asymptotes = []
for color, cid in zip(cmap, ready):
    out = predict(cid, HORIZONS[-1])
    n_axis = np.array(out['n_axis'])
    s_m = np.array(out['soh_trajectory_mean'])
    asymptotes.append((cid, float(s_m[-1])))
    ax.plot(n_axis, s_m, '-', color=color, lw=1.0, alpha=0.85)

for th in [0.80, 0.60, 0.40, 0.20, 0.0]:
    ax.axhline(th, ls='--', color='gray', alpha=0.4, lw=0.8)
    ax.text(HORIZONS[-1]*0.95, th+0.02, f'SoH {th}', fontsize=8, color='gray', ha='right')
ax.set_xscale('symlog', linthresh=100)
ax.set(xlabel='Cycle (symlog)', ylabel='SoH', ylim=(-0.05, 1.05),
        title=f'PINN — all REPT cells, 1 M-cycle horizon (semi-log)')
ax.grid(True, which='both', alpha=0.3)

min_asy = min(a for _, a in asymptotes); max_asy = max(a for _, a in asymptotes)
ax.text(0.02, 0.04,
        f'asymptotic SoH range over {len(ready)} cells: [{min_asy:.4f}, {max_asy:.4f}]',
        transform=ax.transAxes, fontsize=10, color='darkred',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.tight_layout()
out_path = Path('outputs/results/pinn_full_lifespan_all_cells.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'wrote → {out_path}')

asy_df = pd.DataFrame(asymptotes, columns=['cell','SoH_at_1M']).sort_values('SoH_at_1M')
print('\nCell-by-cell asymptotic SoH at 1 M cycles (sorted):')
print(asy_df.to_string(index=False))

## 5. Reading the result

- If the cells all land on an asymptote well above SoH = 0 (likely outcome): the model has **no signal in the training data for late-life ageing**. The constraint `dSoH/dn = −softplus(NN(...))` makes the trajectory monotonically decreasing, but the network output saturates as SoH drops, so the fade rate approaches zero and the trajectory levels off.
- If a few cells do reach 0 at extreme horizons (unlikely): expect the crossing to be far past 100 k cycles, with the model effectively extrapolating a linear or sub-linear tail off the training distribution.
- The instantaneous-rate plot in section 3 shows directly *why* the SoH = 0 endpoint is structurally unreachable for this checkpoint: rate decays as fast as (or faster than) SoH itself.

If you need a model that genuinely predicts to SoH = 0, the architectural fix is to remove the `softplus` constraint or replace it with `−exp(...)` so the rate can grow as SoH drops — then retrain with synthetic trajectories that go all the way to EOL. Both are real model changes, not configuration tweaks.